In [ ]:
from PIL import Image
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import DPTFeatureExtractor, DPTForDepthEstimation
from warnings import simplefilter
simplefilter('ignore')

In [ ]:
import os
import pandas as pd
from pathlib import Path

In [ ]:
import yaml
from ultralytics import YOLO

In [ ]:
# Load Segmentation Model (for Road Detection)
seg_feature_extractor = SegformerFeatureExtractor.from_pretrained("nvidia/segformer-b5-finetuned-cityscapes-1024-1024")
seg_model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b5-finetuned-cityscapes-1024-1024")
# Move segmentation model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seg_model = seg_model.to(device)

# Load Depth Estimation Model
depth_feature_extractor = DPTFeatureExtractor.from_pretrained("Intel/dpt-large")
depth_model = DPTForDepthEstimation.from_pretrained("Intel/dpt-large")
# Move depth model to GPU
depth_model = depth_model.to(device)

def get_road_mask(image_path, target_size):
    """Performs semantic segmentation to detect roads and returns a resized binary mask."""
    image = Image.open(image_path).convert("RGB")
    inputs = seg_feature_extractor(images=image, return_tensors="pt")
    # Move inputs to GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Get segmentation output
    with torch.no_grad():
        outputs = seg_model(**inputs)
    logits = outputs.logits
    # Move logits back to CPU for NumPy operations
    segmentation = logits.argmax(dim=1).squeeze().cpu().numpy()
    
    # Define road class index (based on Cityscapes dataset)
    ROAD_CLASS_INDEX = 0  # Adjust this if needed
    # Create binary mask (255 for roads, 0 for others)
    road_mask = np.where(segmentation == ROAD_CLASS_INDEX, 255, 0).astype(np.uint8)
    # Resize road mask to match depth image size
    road_mask = cv2.resize(road_mask, target_size, interpolation=cv2.INTER_NEAREST)
    return Image.fromarray(road_mask)

def get_depth_map(image_path):
    """Generates an RGB depth map from an image."""
    image = Image.open(image_path)
    pixel_values = depth_feature_extractor(images=image, return_tensors="pt").pixel_values
    # Move inputs to GPU
    pixel_values = pixel_values.to(device)
    
    # Get depth prediction
    with torch.no_grad():
        outputs = depth_model(pixel_values)
        predicted_depth = outputs.predicted_depth
    
    # Resize to match original image (keep on GPU for interpolation)
    prediction = torch.nn.functional.interpolate(
        predicted_depth.unsqueeze(1),
        size=image.size[::-1],
        mode="bicubic",
        align_corners=False,
    )
    
    # Move back to CPU for NumPy operations
    prediction = prediction.squeeze().cpu().numpy()
    
    # Normalize depth values to [0, 255]
    normalized_depth = (prediction - prediction.min()) / (prediction.max() - prediction.min())
    depth_colored = (plt.cm.viridis(normalized_depth) * 255).astype(np.uint8)
    return Image.fromarray(depth_colored[:, :, :3])  # Keep RGB channels

def overlay_road_on_depth(depth_image, road_mask):
    """Overlays the road mask on the depth map, highlighting roads in red."""
    depth_array = np.array(depth_image)
    road_array = np.array(road_mask)
    # Define the road highlight color (soft yellow-green)
    highlight_color = np.array([180, 230, 100])
    # Apply the highlight color where the road is detected
    mask_indices = road_array > 0
    depth_array[mask_indices] = highlight_color
    return Image.fromarray(depth_array)

def process_image(image_path):
    """Full pipeline: Get depth map, detect road, and overlay."""
    depth_map = get_depth_map(image_path)
    
    # Get the depth map size
    target_size = depth_map.size  # (width, height)
    # Get road mask with the correct size
    road_mask = get_road_mask(image_path, target_size)
    # Overlay road on depth
    final_image = overlay_road_on_depth(depth_map, road_mask)
    return final_image

# Add a helper function to check if GPU is available and show memory usage
def print_gpu_info():
    if torch.cuda.is_available():
        print(f"GPU Available: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
        print(f"GPU Memory Reserved: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")
    else:
        print("GPU not available. Running on CPU.")

In [ ]:
class YOLOTrainerDetector:
    def __init__(self):
        self.img_size = 640
        self.conf_thresh = 0.25
        self.epochs = 50
        self.batch_size = 16
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
    def prepare_data(self, annotations_path, data_dir):
        # Create directories
        os.makedirs('dataset/images/train', exist_ok=True)
        os.makedirs('dataset/images/val', exist_ok=True)
        os.makedirs('dataset/labels/train', exist_ok=True)
        os.makedirs('dataset/labels/val', exist_ok=True)
        
        # Read annotations
        df = pd.read_csv(annotations_path)
        
        # Extract unique class names and create mapping
        unique_classes = sorted(df['class'].unique())
        class_dict = {class_name: i for i, class_name in enumerate(unique_classes)}
        
        print(f"Found {len(unique_classes)} classes: {unique_classes}")
        print(f"Class mapping: {class_dict}")
        
        # Create YAML config file for YOLOv8
        data_yaml = {
            'path': os.path.abspath('dataset'),
            'train': 'images/train',
            'val': 'images/val',
            'names': {i: name for i, name in enumerate(unique_classes)}
        }
        
        with open('dataset/data.yaml', 'w') as f:
            yaml.dump(data_yaml, f, sort_keys=False)
        
        # Group by image
        image_groups = df.groupby('filename')
        
        # Split into train (80%) and validation (20%) sets
        image_paths = list(image_groups.groups.keys())
        split_idx = int(len(image_paths) * 0.8)
        train_images = image_paths[:split_idx]
        val_images = image_paths[split_idx:]
        
        # Process each image and its annotations
        for img_path in train_images:
            self.process_image(img_path, image_groups, data_dir, 'train', class_dict)
        
        for img_path in val_images:
            self.process_image(img_path, image_groups, data_dir, 'val', class_dict)
        
        print(f"Dataset prepared with {len(train_images)} training and {len(val_images)} validation images")
        return 'dataset/data.yaml'

    def process_image(self, img_path, image_groups, data_dir, split, class_dict):
        """Process a single image and its annotations"""
        # Get annotations for this image
        annotations = image_groups.get_group(img_path)
        
        # Copy image to dataset
        img_src = os.path.join(data_dir, img_path)
        img_dst = os.path.join('dataset/images', split, img_path)
        
        # Skip if image doesn't exist
        if not os.path.exists(img_src):
            print(f"Warning: Image {img_src} not found, skipping")
            return
        
        # Copy image
        img = Image.open(img_src)
        os.makedirs(os.path.dirname(img_dst), exist_ok=True)
        img.save(img_dst)
        
        # Get image dimensions from CSV
        img_width = annotations['width'].iloc[0]
        img_height = annotations['height'].iloc[0]
        
        # Create label file (YOLO format: class x_center y_center width height)
        label_path = os.path.join('dataset/labels', split, os.path.splitext(img_path)[0] + '.txt')
        os.makedirs(os.path.dirname(label_path), exist_ok=True)
        
        with open(label_path, 'w') as f:
            for _, row in annotations.iterrows():
                # Get class ID from the class column
                class_name = row['class']
                class_id = class_dict[class_name]
                
                # Convert bbox coordinates to YOLO format
                x_min, y_min, x_max, y_max = row['xmin'], row['ymin'], row['xmax'], row['ymax']
                
                # Normalize to 0-1
                x_center = ((x_min + x_max) / 2) / img_width
                y_center = ((y_min + y_max) / 2) / img_height
                bbox_width = (x_max - x_min) / img_width
                bbox_height = (y_max - y_min) / img_height
                
                # Write to file
                f.write(f"{class_id} {x_center} {y_center} {bbox_width} {bbox_height}\n")

    def train_model(self, data_yaml, weights='yolov8n.pt'):
        model = YOLO(weights).to(self.device)
        
        results = model.train(
            data=data_yaml,
            epochs=self.epochs,
            batch=self.batch_size,
            imgsz=self.img_size,
            patience=10,
            save=True,
            device=self.device
        )
        
        best_weights = results.best
        print(f"Training completed. Best weights saved to: {best_weights}")
        return best_weights

    def detect_obstacles(self, model_path, image_path, target_classes=None):
        # Set default target classes if not provided
        if target_classes is None:
            target_classes = ['pedestrian', 'car', 'truck', 'biker']
    
        # Convert target classes to lowercase for case-insensitive comparison
        target_classes = [cls.lower() for cls in target_classes]
    
        model = YOLO(model_path).to(self.device)
    
        # Prepare list for filtered detections
        filtered_boxes = []
    
        # Handle if image_path is a directory
        if os.path.isdir(image_path):
            image_files = [os.path.join(image_path, f) for f in os.listdir(image_path) 
                          if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        else:
            image_files = [image_path]
    
        # Process each image
        for img_file in image_files:
            results = model.predict(
                source=img_file,
                conf=self.conf_thresh,  # Use model's default threshold for detection
                imgsz=self.img_size,
                save=False,
                device=self.device,
                verbose=False
            )
        
            # Load image for visualization (still keeping visualization)
            img = cv2.imread(img_file)
        
            # Process results
            for result in results:
                boxes = result.boxes
            
                for box in boxes:
                    # Get box coordinates
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                    conf = float(box.conf[0].cpu().numpy())
                    cls_id = int(box.cls[0].cpu().numpy())
                    cls_name = result.names[cls_id]
                
                    # Filter by confidence and class
                    if conf >= self.conf_thresh and cls_name.lower() in target_classes:
                        filtered_boxes.append((x1, y1, x2, y2))
        
        return filtered_boxes

In [ ]:
#depth image plus semantic mask
final_image = process_image(r"dataset\images\val\1478900957671553561_jpg.rf.GZVdKZdkyUXLyD18XyCm.jpg")

In [ ]:
#annotation boxes
yolo = YOLOTrainerDetector()
print(yolo.detect_obstacles("runs/detect/train/weights/best.pt", r"D:\minie\code\testing\000153_11.png"))